# 03 Ranking and Explainability
#
Clean Phase 3 notebook.
#
This notebook does five things:
1. Load frozen Phase 1 and Phase 2 outputs
2. Build a master ranking frame from frozen tabular and graph scores
3. Train proposal-aligned attention-based ranking experiments
4. Save raw explainability, robustness, and operational evaluation outputs
5. Write an internal manifest for later `rq*_artifacts.py` scripts
#
Important design notes:
- Phase 1 and Phase 2 models remain frozen
- Phase 3 saves only raw scientific outputs
- No thesis/public figures or tables are generated here
- Human-evaluation artifacts are not fabricated here

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])

In [2]:
import json
import os
import random
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")

## Cell 1: Settings
#
Update the two input roots to match your Kaggle mounted dataset names.

In [3]:
SEED = 42

PHASE1_INPUT_ROOT = Path("/kaggle/input/datasets/ashwinwalunj/ashwin-thesis-outputs/thesis_outputs")
PHASE2_INPUT_ROOT = Path("/kaggle/input/datasets/ashwinwalunj/ashwin-thesis-outputs-phase2/thesis_outputs")

OUTPUT_ROOT = Path("/kaggle/working/thesis_outputs")
PHASE3_ROOT = OUTPUT_ROOT / "artifacts" / "phase03_ranking_explainability"

PHASE1_ARTIFACTS = PHASE1_INPUT_ROOT / "artifacts" / "phase01_data_baselines"
PHASE2_ARTIFACTS = PHASE2_INPUT_ROOT / "artifacts" / "phase02_graph_models"

TARGET_COL = "isFraud"
ID_COL = "TransactionID"
TIME_COL = "TransactionDT"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REFRESH_RANKER = False
RUN_SHAP = True
RUN_GRAPH_CASE_EXPLANATIONS = True
RUN_EXPLANATION_EVALUATION = True
RUN_ROI_ANALYSIS = True

ATTN_HIDDEN_DIM = 32
ATTN_DROPOUT = 0.05
ATTN_LR = 7e-4
ATTN_WEIGHT_DECAY = 1e-5
ATTN_EPOCHS = 50
ATTN_PATIENCE = 8
POS_WEIGHT_CAP = 10.0
ATTN_NOISE_STD = 0.01
ATTN_ENTROPY_WEIGHT = 2e-3
PAIRWISE_LOSS_WEIGHT = 0.35
MAX_PAIRWISE_SAMPLES = 50000

TOP_ALERT_CASES = 12
TOP_GRAPH_EVIDENCE_PER_CASE = 5
TOP_LGBM_FEATURES_PER_CASE = 5
ATTENTION_HEATMAP_CASES = 20
EXPLANATION_SAMPLE_SIZE = 500
NOISE_LEVELS = [0.00, 0.01, 0.03, 0.05]
DAILY_BUDGETS = [50, 100, 250, 500]
SIMULATION_TIME_SERIES_BUDGET = 100

AVG_FRAUD_VALUE_CAPTURED = 250.0
REVIEW_COST_PER_ALERT = 5.0

SOURCE_SCORE_COLUMNS = [
    "lgbm_score",
    "xgb_score",
    "full_TemporalGraphSAGE_score",
    "full_GraphSAGE_score",
]
SOURCE_DISPLAY_NAMES = [
    "LightGBM",
    "XGBoost",
    "TemporalGraphSAGE",
    "GraphSAGE",
]
BASE_CONTEXT_COLUMNS = [
    "transaction_amt_log1p",
    "amount_norm",
    "dt_hour_sin",
    "dt_hour_cos",
    "dt_dayofweek_sin",
    "dt_dayofweek_cos",
]
TEMPORAL_CONTEXT_COLUMNS = [
    "dt_hour_sin",
    "dt_hour_cos",
    "dt_dayofweek_sin",
    "dt_dayofweek_cos",
]
AMOUNT_CONTEXT_COLUMNS = [
    "transaction_amt_log1p",
    "amount_norm",
]
TABULAR_AUX_COLUMNS = [
    "lgbm_score",
    "xgb_score",
    "tabular_mean_score",
    "lgbm_xgb_gap",
]
GRAPH_AUX_COLUMNS = [
    "full_TemporalGraphSAGE_score",
    "full_GraphSAGE_score",
    "graph_mean_score",
    "temporal_graph_gap",
    "tabular_graph_gap",
]

RANKING_METHODS_BASE = [
    ("LightGBM", "lgbm_score"),
    ("XGBoost", "xgb_score"),
    ("TemporalGraphSAGE", "full_TemporalGraphSAGE_score"),
    ("WeightedBlend", "weighted_blend_score"),
]

for folder in [
    PHASE3_ROOT,
    OUTPUT_ROOT / "manifests",
]:
    folder.mkdir(parents=True, exist_ok=True)

## Cell 2: Reproducibility

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Cell 3: Helper Functions

In [5]:
def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def slugify(text):
    return (
        str(text)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )


def precision_at_k(y_true, y_score, k):
    k = min(int(k), len(y_true))
    top_idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[top_idx]))


def recall_at_k(y_true, y_score, k):
    k = min(int(k), len(y_true))
    top_idx = np.argsort(-y_score)[:k]
    positives = max(int(np.sum(y_true)), 1)
    return float(np.sum(y_true[top_idx]) / positives)


def ndcg_at_k(y_true, y_score, k):
    k = min(int(k), len(y_true))
    top_idx = np.argsort(-y_score)[:k]
    gains = y_true[top_idx]
    discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(gains * discounts))

    ideal_gains = np.sort(y_true)[::-1][:k]
    ideal_discounts = 1.0 / np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = float(np.sum(ideal_gains * ideal_discounts))
    if idcg <= 0:
        return 0.0
    return dcg / idcg


def cumulative_gain_curve(y_true, y_score, num_points=100):
    order = np.argsort(-y_score)
    sorted_true = y_true[order]
    positives = max(int(np.sum(sorted_true)), 1)
    fractions = np.linspace(0.01, 1.0, num_points)
    gains = []
    for frac in fractions:
        k = max(int(len(sorted_true) * frac), 1)
        gains.append(float(np.sum(sorted_true[:k]) / positives))
    return pd.DataFrame({"PopulationFraction": fractions, "CumulativeGain": gains})


def simulate_daily_budget(score_df, score_col, budgets):
    rows = []
    for budget in budgets:
        for day_value, day_df in score_df.groupby("day_index"):
            ranked = day_df.sort_values(score_col, ascending=False).head(budget)
            fraud_found = int(ranked["y_true"].sum())
            fraud_total = max(int(day_df["y_true"].sum()), 1)
            rows.append(
                {
                    "Budget": int(budget),
                    "Day": int(day_value),
                    "ScoreColumn": score_col,
                    "AlertsReviewed": int(len(ranked)),
                    "FraudCaptured": fraud_found,
                    "DailyRecall": fraud_found / fraud_total,
                    "DailyPrecision": fraud_found / max(len(ranked), 1),
                }
            )
    return pd.DataFrame(rows)


def add_phase2_style_graph_keys(df):
    out = df.copy()
    out["dt_hour"] = ((out[TIME_COL] // 3600) % 24).astype("int16")
    out["dt_dayofweek"] = ((out[TIME_COL] // 86400) % 7).astype("int16")
    out["day_index"] = (out[TIME_COL] // 86400).astype("int32")
    out["dt_hour_sin"] = np.sin(2 * np.pi * out["dt_hour"].to_numpy(dtype=np.float32) / 24.0)
    out["dt_hour_cos"] = np.cos(2 * np.pi * out["dt_hour"].to_numpy(dtype=np.float32) / 24.0)
    out["dt_dayofweek_sin"] = np.sin(2 * np.pi * out["dt_dayofweek"].to_numpy(dtype=np.float32) / 7.0)
    out["dt_dayofweek_cos"] = np.cos(2 * np.pi * out["dt_dayofweek"].to_numpy(dtype=np.float32) / 7.0)

    out["card_identity_key"] = np.where(
        out["card1"].notna(),
        out[["card1", "card2", "card3", "card5"]].fillna("na").astype(str).agg("_".join, axis=1),
        np.nan,
    )
    out["device_key"] = np.where(
        out["DeviceType"].notna() | out["DeviceInfo"].notna(),
        out[["DeviceType", "DeviceInfo"]].fillna("na").astype(str).agg("_".join, axis=1),
        np.nan,
    )
    out["payer_email_key"] = np.where(
        out["P_emaildomain"].notna(),
        "P_" + out["P_emaildomain"].astype(str),
        np.nan,
    )
    out["receiver_email_key"] = np.where(
        out["R_emaildomain"].notna(),
        "R_" + out["R_emaildomain"].astype(str),
        np.nan,
    )
    out["merchant_proxy_key"] = np.where(
        out["ProductCD"].notna() & out["addr1"].notna() & out["addr2"].notna(),
        out[["ProductCD", "addr1", "addr2"]].astype(str).agg("_".join, axis=1),
        np.nan,
    )

    train_mask = out["split"] == "train"
    train_amt = out.loc[train_mask, "TransactionAmt"].fillna(0).clip(lower=0)
    train_amt_log = np.log1p(train_amt)
    amount_low = float(train_amt_log.quantile(0.01))
    amount_high = float(train_amt_log.quantile(0.99))
    if amount_high <= amount_low:
        amount_high = amount_low + 1.0

    out["transaction_amt_log1p"] = np.log1p(out["TransactionAmt"].fillna(0).clip(lower=0))
    out["amount_norm"] = (
        (out["transaction_amt_log1p"] - amount_low) / (amount_high - amount_low)
    ).clip(0.0, 1.0)
    out["amount_norm"] = (0.1 + 0.9 * out["amount_norm"]).astype("float32")
    return out


def fit_standardization(train_df, columns):
    means = train_df[columns].mean()
    stds = train_df[columns].std().replace(0, 1.0).fillna(1.0)
    return means.to_dict(), stds.to_dict()


def apply_standardization(df, columns, means, stds):
    out = df.copy()
    for col in columns:
        out[col] = (out[col] - means[col]) / stds[col]
    out[columns] = out[columns].fillna(0.0)
    return out


def build_branch_array(df, columns):
    if columns:
        return df[columns].to_numpy(dtype=np.float32)
    return np.zeros((len(df), 1), dtype=np.float32)


def build_attention_arrays(df, source_cols, context_cols, tabular_aux_cols, graph_aux_cols):
    source_scores = df[source_cols].to_numpy(dtype=np.float32)
    centered_scores = source_scores - source_scores.mean(axis=1, keepdims=True)
    source_array = np.stack([source_scores, centered_scores.astype(np.float32)], axis=2)
    context_array = build_branch_array(df, context_cols)
    tabular_aux_array = build_branch_array(df, tabular_aux_cols)
    graph_aux_array = build_branch_array(df, graph_aux_cols)
    return source_array, context_array, tabular_aux_array, graph_aux_array


def truncate_label(value, max_len=32):
    text = str(value)
    if len(text) <= max_len:
        return text
    return text[: max_len - 3] + "..."


def choose_reference_case(case_df):
    positive_cases = case_df[case_df["y_true"] == 1]
    if not positive_cases.empty:
        return positive_cases.iloc[0]
    return case_df.iloc[0]


def compute_attention_support_size(attention_matrix, threshold=0.80):
    sizes = []
    for row in attention_matrix:
        order = np.sort(row)[::-1]
        cum = np.cumsum(order)
        sizes.append(int(np.searchsorted(cum, threshold) + 1))
    return np.array(sizes, dtype=int)


def run_attention_forward(model, source_array, context_array, tabular_aux_array, graph_aux_array):
    source_t = torch.tensor(source_array, dtype=torch.float32, device=DEVICE)
    context_t = torch.tensor(context_array, dtype=torch.float32, device=DEVICE)
    tabular_aux_t = torch.tensor(tabular_aux_array, dtype=torch.float32, device=DEVICE)
    graph_aux_t = torch.tensor(graph_aux_array, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits, attention = model(source_t, context_t, tabular_aux_t, graph_aux_t)
    return torch.sigmoid(logits).cpu().numpy(), attention.cpu().numpy()


def explain_top_cases_with_graph_context(case_df, graph_lookup_df, edge_df, entity_df):
    txn_lookup = graph_lookup_df.set_index(ID_COL)
    edge_lookup = edge_df.groupby("src")
    entity_lookup = entity_df.set_index("node_id")

    rows = []
    for rank_idx, row in case_df.reset_index(drop=True).iterrows():
        txn_id = int(row[ID_COL])
        if txn_id not in txn_lookup.index:
            continue
        txn_info = txn_lookup.loc[txn_id]
        txn_node_id = int(txn_info["txn_idx"])
        if txn_node_id not in edge_lookup.groups:
            continue

        local_edges = edge_lookup.get_group(txn_node_id).copy().sort_values("weight", ascending=False)
        local_edges = local_edges.head(TOP_GRAPH_EVIDENCE_PER_CASE)

        for _, edge_row in local_edges.iterrows():
            dst_node = int(edge_row["dst"])
            if dst_node not in entity_lookup.index:
                continue
            entity_row = entity_lookup.loc[dst_node]
            rows.append(
                {
                    ID_COL: txn_id,
                    "AlertRank": int(rank_idx + 1),
                    "EntityType": entity_row["entity_type"],
                    "EntityKey": entity_row["entity_key"],
                    "EdgeWeight": float(edge_row["weight"]),
                    "EntityTrainFraudRate": float(entity_row["train_fraud_rate"]),
                }
            )
    return pd.DataFrame(rows)


def top_lgbm_features_for_cases(case_feature_df, case_ids, case_lookup, feature_schema, lgbm_model):
    explainer = shap.TreeExplainer(lgbm_model.booster_)
    feature_block = case_feature_df[feature_schema["feature_columns"]].copy()
    shap_values = explainer.shap_values(feature_block)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]

    rows = []
    for row_idx, txn_id in enumerate(case_ids):
        top_idx = np.argsort(-np.abs(shap_values[row_idx]))[:TOP_LGBM_FEATURES_PER_CASE]
        for feature_rank, feat_idx in enumerate(top_idx, start=1):
            rows.append(
                {
                    ID_COL: int(txn_id),
                    TIME_COL: int(case_lookup[txn_id][TIME_COL]),
                    "FeatureRank": int(feature_rank),
                    "Feature": feature_schema["feature_columns"][feat_idx],
                    "FeatureValue": float(feature_block.iloc[row_idx, feat_idx]),
                    "SHAPValue": float(shap_values[row_idx, feat_idx]),
                }
            )
    return pd.DataFrame(rows), shap_values


def build_case_summary_table(top_case_df, lgbm_case_feature_df, graph_case_df, source_score_columns, source_display_names):
    rows = []
    for _, case_row in top_case_df.iterrows():
        txn_id = int(case_row[ID_COL])
        feature_rows = lgbm_case_feature_df[lgbm_case_feature_df[ID_COL] == txn_id].sort_values("FeatureRank")
        graph_rows = graph_case_df[graph_case_df[ID_COL] == txn_id].sort_values("EdgeWeight", ascending=False)

        top_feature_name = feature_rows.iloc[0]["Feature"] if not feature_rows.empty else None
        top_feature_shap = float(feature_rows.iloc[0]["SHAPValue"]) if not feature_rows.empty else np.nan
        top_graph_entity_type = graph_rows.iloc[0]["EntityType"] if not graph_rows.empty else None
        top_graph_entity_key = graph_rows.iloc[0]["EntityKey"] if not graph_rows.empty else None
        top_graph_edge_weight = float(graph_rows.iloc[0]["EdgeWeight"]) if not graph_rows.empty else np.nan

        attention_weights = [float(case_row[f"{col}_attention"]) for col in source_score_columns]
        top_source_idx = int(np.argmax(attention_weights))

        rows.append(
            {
                ID_COL: txn_id,
                TIME_COL: int(case_row[TIME_COL]),
                "TrueLabel": int(case_row["y_true"]),
                "AttentionRankerScore": float(case_row["attention_ranker_score"]),
                "TopAttentionSource": source_display_names[top_source_idx],
                "TopAttentionWeight": float(attention_weights[top_source_idx]),
                "TopLightGBMFeature": top_feature_name,
                "TopLightGBMSHAP": top_feature_shap,
                "TopGraphEntityType": top_graph_entity_type,
                "TopGraphEntityKey": top_graph_entity_key,
                "TopGraphEdgeWeight": top_graph_edge_weight,
                "ReviewDecision": "FlagForAnalystReview",
            }
        )
    return pd.DataFrame(rows)


def compute_lgbm_mask_curve(feature_df, shap_values, medians, feature_schema, lgbm_model, max_k=4):
    base_scores = lgbm_model.predict_proba(feature_df)[:, 1]
    top_orders = np.argsort(-np.abs(shap_values), axis=1)
    rows = [{"ComponentsMasked": 0, "MaskingStrategy": "Top", "ScoreRetention": 1.0, "Method": "LightGBM"}]
    rows.append({"ComponentsMasked": 0, "MaskingStrategy": "Random", "ScoreRetention": 1.0, "Method": "LightGBM"})

    for k in range(1, max_k + 1):
        top_masked_scores = []
        random_masked_scores = []
        for row_idx in range(len(feature_df)):
            selected_top = top_orders[row_idx][:k]
            selected_random = np.random.permutation(feature_df.shape[1])[:k]

            masked_top = feature_df.iloc[[row_idx]].copy()
            masked_random = feature_df.iloc[[row_idx]].copy()

            for feat_idx in selected_top:
                feat_name = feature_schema["feature_columns"][feat_idx]
                masked_top[feat_name] = medians[feat_name]
            for feat_idx in selected_random:
                feat_name = feature_schema["feature_columns"][feat_idx]
                masked_random[feat_name] = medians[feat_name]

            top_masked_scores.append(float(lgbm_model.predict_proba(masked_top)[:, 1][0]))
            random_masked_scores.append(float(lgbm_model.predict_proba(masked_random)[:, 1][0]))

        rows.append(
            {
                "ComponentsMasked": int(k),
                "MaskingStrategy": "Top",
                "ScoreRetention": float(np.mean(top_masked_scores) / max(np.mean(base_scores), 1e-6)),
                "Method": "LightGBM",
            }
        )
        rows.append(
            {
                "ComponentsMasked": int(k),
                "MaskingStrategy": "Random",
                "ScoreRetention": float(np.mean(random_masked_scores) / max(np.mean(base_scores), 1e-6)),
                "Method": "LightGBM",
            }
        )
    return pd.DataFrame(rows), base_scores


def compute_attention_mask_curve(
    model,
    source_array,
    context_array,
    tabular_aux_array,
    graph_aux_array,
    attention_weights,
    max_k=4,
):
    base_scores, _ = run_attention_forward(model, source_array, context_array, tabular_aux_array, graph_aux_array)
    rows = [{"ComponentsMasked": 0, "MaskingStrategy": "Top", "ScoreRetention": 1.0, "Method": "AttentionRanker"}]
    rows.append({"ComponentsMasked": 0, "MaskingStrategy": "Random", "ScoreRetention": 1.0, "Method": "AttentionRanker"})

    for k in range(1, max_k + 1):
        source_top = source_array.copy()
        source_random = source_array.copy()

        for row_idx in range(source_array.shape[0]):
            chosen_top = np.argsort(-attention_weights[row_idx])[:k]
            chosen_random = np.random.permutation(source_array.shape[1])[:k]
            source_top[row_idx, chosen_top, :] = 0.0
            source_random[row_idx, chosen_random, :] = 0.0

        top_scores, _ = run_attention_forward(model, source_top, context_array, tabular_aux_array, graph_aux_array)
        random_scores, _ = run_attention_forward(model, source_random, context_array, tabular_aux_array, graph_aux_array)

        rows.append(
            {
                "ComponentsMasked": int(k),
                "MaskingStrategy": "Top",
                "ScoreRetention": float(np.mean(top_scores) / max(np.mean(base_scores), 1e-6)),
                "Method": "AttentionRanker",
            }
        )
        rows.append(
            {
                "ComponentsMasked": int(k),
                "MaskingStrategy": "Random",
                "ScoreRetention": float(np.mean(random_scores) / max(np.mean(base_scores), 1e-6)),
                "Method": "AttentionRanker",
            }
        )
    return pd.DataFrame(rows), base_scores


class AttentionFusionRanker(nn.Module):
    def __init__(
        self,
        num_sources,
        source_feat_dim,
        context_dim,
        tabular_aux_dim,
        graph_aux_dim,
        hidden_dim,
        dropout,
        use_graph_branch=True,
    ):
        super().__init__()
        self.use_graph_branch = use_graph_branch
        self.source_emb = nn.Parameter(torch.randn(num_sources, hidden_dim) * 0.02)
        self.source_proj = nn.Linear(source_feat_dim, hidden_dim)
        self.context_proj = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.source_gate = nn.Sequential(
            nn.Linear(context_dim + tabular_aux_dim + graph_aux_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_sources),
        )
        self.tabular_proj = nn.Sequential(
            nn.Linear(tabular_aux_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )
        if self.use_graph_branch:
            self.graph_proj = nn.Sequential(
                nn.Linear(graph_aux_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.graph_gate = nn.Sequential(
                nn.Linear(context_dim + tabular_aux_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )
            nn.init.constant_(self.graph_gate[-1].bias, -1.5)
        else:
            self.graph_proj = None
            self.graph_gate = None
        self.attn_source = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_context = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_v = nn.Linear(hidden_dim, 1, bias=False)
        self.output = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, source_feats, context_feats, tabular_aux_feats, graph_aux_feats):
        source_tokens = self.source_proj(source_feats) + self.source_emb.unsqueeze(0)
        source_gate = torch.sigmoid(
            self.source_gate(torch.cat([context_feats, tabular_aux_feats, graph_aux_feats], dim=1))
        )
        source_tokens = source_tokens * source_gate.unsqueeze(-1)
        context_hidden = self.context_proj(context_feats)
        tabular_hidden = self.tabular_proj(tabular_aux_feats)
        if self.use_graph_branch:
            graph_hidden = self.graph_proj(graph_aux_feats)
            graph_gate = torch.sigmoid(self.graph_gate(torch.cat([context_feats, tabular_aux_feats], dim=1)))
            graph_hidden = graph_hidden * graph_gate
        else:
            graph_hidden = torch.zeros_like(tabular_hidden)

        attn_hidden = torch.tanh(
            self.attn_source(source_tokens) + self.attn_context(context_hidden).unsqueeze(1)
        )
        attn_weights = torch.softmax(self.attn_v(attn_hidden).squeeze(-1), dim=1)
        fused_sources = torch.sum(attn_weights.unsqueeze(-1) * source_tokens, dim=1)

        raw_scores = source_feats[:, :, 0]
        weighted_score = torch.sum(attn_weights * raw_scores, dim=1, keepdim=True)
        mean_score = raw_scores.mean(dim=1, keepdim=True)
        std_score = raw_scores.std(dim=1, keepdim=True, unbiased=False)

        output_input = torch.cat(
            [fused_sources, context_hidden, tabular_hidden, graph_hidden, weighted_score, mean_score, std_score],
            dim=1,
        )
        logits = self.output(output_input).squeeze(-1)
        return logits, attn_weights


def rebuild_attention_model(schema):
    context_dim = len(schema["context_columns"]) if schema["context_columns"] else 1
    tabular_aux_dim = len(schema["tabular_aux_columns"]) if schema["tabular_aux_columns"] else 1
    graph_aux_dim = len(schema["graph_aux_columns"]) if schema["graph_aux_columns"] else 1
    return AttentionFusionRanker(
        num_sources=len(schema["source_columns"]),
        source_feat_dim=2,
        context_dim=context_dim,
        tabular_aux_dim=tabular_aux_dim,
        graph_aux_dim=graph_aux_dim,
        hidden_dim=schema["hidden_dim"],
        dropout=schema["dropout"],
        use_graph_branch=schema["use_graph_branch"],
    ).to(DEVICE)


def pairwise_ranking_loss(logits, y_true, max_pairs=MAX_PAIRWISE_SAMPLES):
    pos_idx = torch.where(y_true > 0.5)[0]
    neg_idx = torch.where(y_true <= 0.5)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return torch.tensor(0.0, device=logits.device)

    num_pairs = min(max_pairs, len(pos_idx) * 4, len(neg_idx) * 4)
    pos_sample = pos_idx[torch.randint(len(pos_idx), (num_pairs,), device=logits.device)]
    neg_sample = neg_idx[torch.randint(len(neg_idx), (num_pairs,), device=logits.device)]
    margin = logits[pos_sample] - logits[neg_sample]
    return F.softplus(-margin).mean()


def train_attention_model(
    model,
    source_train,
    context_train,
    tabular_aux_train,
    graph_aux_train,
    y_train,
    source_valid,
    context_valid,
    tabular_aux_valid,
    graph_aux_valid,
    y_valid,
    pos_weight,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=ATTN_LR, weight_decay=ATTN_WEIGHT_DECAY)
    best_ap = -1.0
    best_epoch = 0
    stale_epochs = 0
    best_state = None
    history_rows = []

    source_train_t = torch.tensor(source_train, dtype=torch.float32, device=DEVICE)
    context_train_t = torch.tensor(context_train, dtype=torch.float32, device=DEVICE)
    tabular_aux_train_t = torch.tensor(tabular_aux_train, dtype=torch.float32, device=DEVICE)
    graph_aux_train_t = torch.tensor(graph_aux_train, dtype=torch.float32, device=DEVICE)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)
    source_valid_t = torch.tensor(source_valid, dtype=torch.float32, device=DEVICE)
    context_valid_t = torch.tensor(context_valid, dtype=torch.float32, device=DEVICE)
    tabular_aux_valid_t = torch.tensor(tabular_aux_valid, dtype=torch.float32, device=DEVICE)
    graph_aux_valid_t = torch.tensor(graph_aux_valid, dtype=torch.float32, device=DEVICE)
    pos_weight_t = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)

    for epoch in range(1, ATTN_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        noisy_source_train_t = source_train_t + torch.randn_like(source_train_t) * ATTN_NOISE_STD
        noisy_context_train_t = context_train_t + torch.randn_like(context_train_t) * ATTN_NOISE_STD
        noisy_tabular_aux_train_t = tabular_aux_train_t + torch.randn_like(tabular_aux_train_t) * ATTN_NOISE_STD
        noisy_graph_aux_train_t = graph_aux_train_t + torch.randn_like(graph_aux_train_t) * ATTN_NOISE_STD

        train_logits, train_attention = model(
            noisy_source_train_t,
            noisy_context_train_t,
            noisy_tabular_aux_train_t,
            noisy_graph_aux_train_t,
        )
        bce_loss = F.binary_cross_entropy_with_logits(train_logits, y_train_t, pos_weight=pos_weight_t)
        rank_loss = pairwise_ranking_loss(train_logits, y_train_t)
        attention_entropy = -(train_attention * torch.log(train_attention.clamp_min(1e-8))).sum(dim=1).mean()
        train_loss = bce_loss + PAIRWISE_LOSS_WEIGHT * rank_loss + ATTN_ENTROPY_WEIGHT * attention_entropy
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            valid_logits, _ = model(source_valid_t, context_valid_t, tabular_aux_valid_t, graph_aux_valid_t)
            valid_probs = torch.sigmoid(valid_logits).detach().cpu().numpy()
            valid_ap = average_precision_score(y_valid, valid_probs)
            valid_roc = roc_auc_score(y_valid, valid_probs)

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": float(train_loss.item()),
                "train_bce_loss": float(bce_loss.item()),
                "train_pairwise_loss": float(rank_loss.item()),
                "valid_auc_pr": float(valid_ap),
                "valid_auc_roc": float(valid_roc),
            }
        )
        print(
            f"AttentionRanker | epoch {epoch:02d} | "
            f"train_loss={float(train_loss.item()):.4f} | valid_auc_pr={float(valid_ap):.4f}"
        )

        if valid_ap > best_ap:
            best_ap = float(valid_ap)
            best_epoch = epoch
            stale_epochs = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1

        if stale_epochs >= ATTN_PATIENCE:
            print(f"Early stopping AttentionRanker at epoch {epoch}")
            break

    return best_state, best_epoch, pd.DataFrame(history_rows)


def fit_attention_experiment(
    experiment_name,
    meta_train_df,
    rank_valid_df,
    rank_test_df,
    source_score_cols,
    context_cols,
    tabular_aux_cols,
    graph_aux_cols,
):
    meta_train_df = meta_train_df.copy().reset_index(drop=True)
    meta_valid_df = rank_valid_df.copy().reset_index(drop=True)

    standardize_cols = []
    for col_group in [context_cols, tabular_aux_cols, graph_aux_cols]:
        for col in col_group:
            if col not in standardize_cols:
                standardize_cols.append(col)

    if standardize_cols:
        feature_means, feature_stds = fit_standardization(meta_train_df, standardize_cols)
        meta_train_std = apply_standardization(meta_train_df, standardize_cols, feature_means, feature_stds)
        meta_valid_std = apply_standardization(meta_valid_df, standardize_cols, feature_means, feature_stds)
        valid_full_std = apply_standardization(rank_valid_df.copy().reset_index(drop=True), standardize_cols, feature_means, feature_stds)
        test_std_df = apply_standardization(rank_test_df.copy().reset_index(drop=True), standardize_cols, feature_means, feature_stds)
    else:
        feature_means, feature_stds = {}, {}
        meta_train_std = meta_train_df.copy()
        meta_valid_std = meta_valid_df.copy()
        valid_full_std = rank_valid_df.copy().reset_index(drop=True)
        test_std_df = rank_test_df.copy().reset_index(drop=True)

    train_source, train_context, train_tabular_aux, train_graph_aux = build_attention_arrays(
        meta_train_std,
        source_score_cols,
        context_cols,
        tabular_aux_cols,
        graph_aux_cols,
    )
    valid_source, valid_context, valid_tabular_aux, valid_graph_aux = build_attention_arrays(
        meta_valid_std,
        source_score_cols,
        context_cols,
        tabular_aux_cols,
        graph_aux_cols,
    )
    valid_full_source, valid_full_context, valid_full_tabular_aux, valid_full_graph_aux = build_attention_arrays(
        valid_full_std,
        source_score_cols,
        context_cols,
        tabular_aux_cols,
        graph_aux_cols,
    )
    test_source, test_context, test_tabular_aux, test_graph_aux = build_attention_arrays(
        test_std_df,
        source_score_cols,
        context_cols,
        tabular_aux_cols,
        graph_aux_cols,
    )

    y_meta_train = meta_train_std["y_true"].astype(int).to_numpy()
    y_meta_valid = meta_valid_std["y_true"].astype(int).to_numpy()
    y_valid_full = valid_full_std["y_true"].astype(int).to_numpy()
    y_test = test_std_df["y_true"].astype(int).to_numpy()

    raw_pos_weight = (len(y_meta_train) - y_meta_train.sum()) / max(y_meta_train.sum(), 1)
    pos_weight = min(float(raw_pos_weight), POS_WEIGHT_CAP)

    context_dim = len(context_cols) if context_cols else 1
    tabular_aux_dim = len(tabular_aux_cols) if tabular_aux_cols else 1
    graph_aux_dim = len(graph_aux_cols) if graph_aux_cols else 1
    model = AttentionFusionRanker(
        num_sources=len(source_score_cols),
        source_feat_dim=2,
        context_dim=context_dim,
        tabular_aux_dim=tabular_aux_dim,
        graph_aux_dim=graph_aux_dim,
        hidden_dim=ATTN_HIDDEN_DIM,
        dropout=ATTN_DROPOUT,
        use_graph_branch=bool(graph_aux_cols),
    ).to(DEVICE)

    best_state, best_epoch, history_df = train_attention_model(
        model=model,
        source_train=train_source,
        context_train=train_context,
        tabular_aux_train=train_tabular_aux,
        graph_aux_train=train_graph_aux,
        y_train=y_meta_train,
        source_valid=valid_source,
        context_valid=valid_context,
        tabular_aux_valid=valid_tabular_aux,
        graph_aux_valid=valid_graph_aux,
        y_valid=y_meta_valid,
        pos_weight=pos_weight,
    )

    final_model = AttentionFusionRanker(
        num_sources=len(source_score_cols),
        source_feat_dim=2,
        context_dim=context_dim,
        tabular_aux_dim=tabular_aux_dim,
        graph_aux_dim=graph_aux_dim,
        hidden_dim=ATTN_HIDDEN_DIM,
        dropout=ATTN_DROPOUT,
        use_graph_branch=bool(graph_aux_cols),
    ).to(DEVICE)
    final_model.load_state_dict(best_state)
    final_model.eval()

    valid_scores, valid_attention = run_attention_forward(
        final_model,
        valid_full_source,
        valid_full_context,
        valid_full_tabular_aux,
        valid_full_graph_aux,
    )
    test_scores, test_attention = run_attention_forward(
        final_model,
        test_source,
        test_context,
        test_tabular_aux,
        test_graph_aux,
    )

    summary = {
        "experiment_name": experiment_name,
        "source_columns": source_score_cols,
        "context_columns": context_cols,
        "tabular_aux_columns": tabular_aux_cols,
        "graph_aux_columns": graph_aux_cols,
        "feature_means": feature_means,
        "feature_stds": feature_stds,
        "hidden_dim": ATTN_HIDDEN_DIM,
        "dropout": ATTN_DROPOUT,
        "use_graph_branch": bool(graph_aux_cols),
        "best_epoch": int(best_epoch),
        "valid_auc_pr": float(average_precision_score(y_valid_full, valid_scores)),
        "valid_auc_roc": float(roc_auc_score(y_valid_full, valid_scores)),
        "test_precision_at_100": float(precision_at_k(y_test, test_scores, 100)),
        "test_recall_at_100": float(recall_at_k(y_test, test_scores, 100)),
        "test_ndcg_at_100": float(ndcg_at_k(y_test, test_scores, 100)),
        "test_auc_pr": float(average_precision_score(y_test, test_scores)),
        "test_auc_roc": float(roc_auc_score(y_test, test_scores)),
    }

    return {
        "model": final_model,
        "history_df": history_df,
        "summary": summary,
        "valid_scores": valid_scores,
        "test_scores": test_scores,
        "valid_attention": valid_attention,
        "test_attention": test_attention,
    }

## Cell 4: Load Frozen Phase 1 And Phase 2 Artifacts

In [6]:
required_phase1_files = [
    PHASE1_ARTIFACTS / "merged_train.parquet",
    PHASE1_ARTIFACTS / "train.parquet",
    PHASE1_ARTIFACTS / "valid.parquet",
    PHASE1_ARTIFACTS / "test.parquet",
    PHASE1_ARTIFACTS / "feature_schema.json",
    PHASE1_ARTIFACTS / "baseline_predictions.parquet",
    PHASE1_ARTIFACTS / "baseline_oof_predictions.parquet",
    PHASE1_ARTIFACTS / "baseline_metrics.json",
    PHASE1_ARTIFACTS / "lgbm_model.pkl",
]
required_phase2_files = [
    PHASE2_ARTIFACTS / "rq1_predictions.parquet",
    PHASE2_ARTIFACTS / "graph_oof_predictions.parquet",
    PHASE2_ARTIFACTS / "full_TemporalGraphSAGE_summary.json",
    PHASE2_ARTIFACTS / "full_GraphSAGE_summary.json",
    PHASE2_ARTIFACTS / "full_edges.parquet",
    PHASE2_ARTIFACTS / "full_entity_nodes.parquet",
]

for file_path in required_phase1_files + required_phase2_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing frozen input file: {file_path}")

raw_merged_df = pd.read_parquet(PHASE1_ARTIFACTS / "merged_train.parquet")
train_df = pd.read_parquet(PHASE1_ARTIFACTS / "train.parquet")
valid_df = pd.read_parquet(PHASE1_ARTIFACTS / "valid.parquet")
test_df = pd.read_parquet(PHASE1_ARTIFACTS / "test.parquet")
feature_schema = load_json(PHASE1_ARTIFACTS / "feature_schema.json")
phase1_metrics = load_json(PHASE1_ARTIFACTS / "baseline_metrics.json")
lgbm_model = joblib.load(PHASE1_ARTIFACTS / "lgbm_model.pkl")
phase1_oof_predictions = pd.read_parquet(PHASE1_ARTIFACTS / "baseline_oof_predictions.parquet")

phase2_predictions = pd.read_parquet(PHASE2_ARTIFACTS / "rq1_predictions.parquet")
phase2_oof_predictions = pd.read_parquet(PHASE2_ARTIFACTS / "graph_oof_predictions.parquet")
phase2_temporal_summary = load_json(PHASE2_ARTIFACTS / "full_TemporalGraphSAGE_summary.json")
phase2_graphsage_summary = load_json(PHASE2_ARTIFACTS / "full_GraphSAGE_summary.json")
full_edge_df = pd.read_parquet(PHASE2_ARTIFACTS / "full_edges.parquet")
full_entity_df = pd.read_parquet(PHASE2_ARTIFACTS / "full_entity_nodes.parquet")

print("Phase 1 raw merged:", raw_merged_df.shape)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)
print("Phase 2 predictions:", phase2_predictions.shape)

Phase 1 raw merged: (590540, 434)
Train: (442905, 479)
Valid: (59054, 479)
Test : (88581, 479)
Phase 2 predictions: (147635, 9)


## Cell 5: Build Phase 3 Master Ranking Frame

In [8]:
graph_lookup_df = raw_merged_df.copy().sort_values(TIME_COL).reset_index(drop=True)
graph_lookup_df["txn_idx"] = np.arange(len(graph_lookup_df), dtype=np.int64)
graph_lookup_df["split"] = "train"
graph_lookup_df.loc[len(train_df): len(train_df) + len(valid_df) - 1, "split"] = "valid"
graph_lookup_df.loc[len(train_df) + len(valid_df):, "split"] = "test"
graph_lookup_df = add_phase2_style_graph_keys(graph_lookup_df)

eval_context_cols = [
    ID_COL,
    TIME_COL,
    TARGET_COL,
    "split",
    "TransactionAmt",
    "transaction_amt_log1p",
    "dt_hour",
    "dt_dayofweek",
    "dt_hour_sin",
    "dt_hour_cos",
    "dt_dayofweek_sin",
    "dt_dayofweek_cos",
    "day_index",
    "card_identity_key",
    "device_key",
    "payer_email_key",
    "receiver_email_key",
    "merchant_proxy_key",
    "amount_norm",
]

phase3_df = phase2_predictions.merge(
    graph_lookup_df[eval_context_cols],
    on=[ID_COL, TIME_COL, "split"],
    how="left",
)

graph_lookup_oof = graph_lookup_df[graph_lookup_df["split"] == "train"].copy()
graph_lookup_oof["split"] = "train_oof"

meta_train_df = phase1_oof_predictions.merge(
    phase2_oof_predictions,
    on=[ID_COL, TIME_COL, "split", "fold_id", "y_true"],
    how="inner",
).merge(
    graph_lookup_oof[eval_context_cols],
    on=[ID_COL, TIME_COL, "split"],
    how="left",
)

if phase3_df[SOURCE_SCORE_COLUMNS].isnull().any().any():
    raise ValueError("Missing required frozen score columns after Phase 3 merge.")
if meta_train_df[SOURCE_SCORE_COLUMNS].isnull().any().any():
    raise ValueError("Missing required OOF score columns after Phase 3 merge.")
if phase3_df[["TransactionAmt", TARGET_COL]].isnull().any().any():
    raise ValueError("Phase 3 context merge is incomplete.")
if meta_train_df[["TransactionAmt", TARGET_COL]].isnull().any().any():
    raise ValueError("Phase 3 OOF context merge is incomplete.")
if not np.array_equal(
    phase3_df["y_true"].astype(int).to_numpy(),
    phase3_df[TARGET_COL].astype(int).to_numpy(),
):
    raise ValueError("Frozen prediction labels do not match raw transaction labels.")
if not np.array_equal(
    meta_train_df["y_true"].astype(int).to_numpy(),
    meta_train_df[TARGET_COL].astype(int).to_numpy(),
):
    raise ValueError("OOF labels do not match raw transaction labels.")

for work_df in [meta_train_df, phase3_df]:
    work_df["score_mean"] = work_df[SOURCE_SCORE_COLUMNS].mean(axis=1)
    work_df["score_std"] = work_df[SOURCE_SCORE_COLUMNS].std(axis=1).fillna(0.0)
    work_df["tabular_mean_score"] = work_df[["lgbm_score", "xgb_score"]].mean(axis=1)
    work_df["graph_mean_score"] = work_df[["full_TemporalGraphSAGE_score", "full_GraphSAGE_score"]].mean(axis=1)
    work_df["lgbm_xgb_gap"] = work_df["lgbm_score"] - work_df["xgb_score"]
    work_df["temporal_graph_gap"] = work_df["full_TemporalGraphSAGE_score"] - work_df["full_GraphSAGE_score"]
    work_df["tabular_graph_gap"] = work_df["tabular_mean_score"] - work_df["graph_mean_score"]

rank_valid_df = phase3_df[phase3_df["split"] == "valid"].copy().sort_values(TIME_COL).reset_index(drop=True)
rank_test_df = phase3_df[phase3_df["split"] == "test"].copy().sort_values(TIME_COL).reset_index(drop=True)
meta_train_df = meta_train_df.sort_values(TIME_COL).reset_index(drop=True)

test_feature_lookup = test_df[[ID_COL] + feature_schema["feature_columns"]].copy()
test_feature_lookup = test_feature_lookup.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)

phase3_df.to_parquet(PHASE3_ROOT / "phase3_master_frame.parquet", index=False)
meta_train_df.to_parquet(PHASE3_ROOT / "phase3_meta_train_frame.parquet", index=False)
print("Phase 3 master frame:", phase3_df.shape)
print("Phase 3 meta-train frame:", meta_train_df.shape)
print("Ranking validation rows:", rank_valid_df.shape)
print("Ranking test rows:", rank_test_df.shape)

Phase 3 master frame: (147635, 32)
Phase 3 meta-train frame: (265233, 32)
Ranking validation rows: (59054, 32)
Ranking test rows: (88581, 32)


## Cell 6: Ranking Baselines

In [9]:
validation_ap_weights = {
    "lgbm_score": phase1_metrics["valid_lgbm_auc_pr"],
    "xgb_score": phase1_metrics["valid_xgb_auc_pr"],
    "full_TemporalGraphSAGE_score": phase2_temporal_summary["valid_auc_pr"],
    "full_GraphSAGE_score": phase2_graphsage_summary["valid_auc_pr"],
}
weight_total = sum(validation_ap_weights.values())
heuristic_weights = {key: value / weight_total for key, value in validation_ap_weights.items()}

for df_ in [rank_valid_df, rank_test_df]:
    df_["weighted_blend_score"] = sum(df_[key] * weight for key, weight in heuristic_weights.items())

save_json(heuristic_weights, PHASE3_ROOT / "heuristic_weights.json")
heuristic_weights

{'lgbm_score': 0.29128474712856117,
 'xgb_score': 0.2895624079289795,
 'full_TemporalGraphSAGE_score': 0.20559825419288297,
 'full_GraphSAGE_score': 0.21355459074957625}

## Cell 7: Attention-Based Fusion Experiments

In [10]:
ATTENTION_MODEL_PATH = PHASE3_ROOT / "attention_ranker_state.pt"
ATTENTION_SCHEMA_PATH = PHASE3_ROOT / "attention_ranker_schema.json"
ATTENTION_HISTORY_PATH = PHASE3_ROOT / "attention_ranker_history.csv"
ATTENTION_SUMMARY_PATH = PHASE3_ROOT / "attention_ranker_summary.json"
ATTENTION_PREDICTIONS_PATH = PHASE3_ROOT / "attention_ranker_predictions.parquet"
ATTENTION_ABLATION_PATH = PHASE3_ROOT / "attention_ablation_raw.csv"
ATTENTION_EXPERIMENT_SUMMARIES_PATH = PHASE3_ROOT / "attention_experiment_summaries.json"

experiment_specs = [
    {
        "experiment_name": "FullModel",
        "source_columns": SOURCE_SCORE_COLUMNS,
        "context_columns": BASE_CONTEXT_COLUMNS,
        "tabular_aux_columns": TABULAR_AUX_COLUMNS,
        "graph_aux_columns": GRAPH_AUX_COLUMNS,
    },
    {
        "experiment_name": "RemoveNeighborhood",
        "source_columns": ["lgbm_score", "xgb_score"],
        "context_columns": BASE_CONTEXT_COLUMNS,
        "tabular_aux_columns": TABULAR_AUX_COLUMNS,
        "graph_aux_columns": [],
    },
    {
        "experiment_name": "RemoveTemporal",
        "source_columns": SOURCE_SCORE_COLUMNS,
        "context_columns": AMOUNT_CONTEXT_COLUMNS,
        "tabular_aux_columns": TABULAR_AUX_COLUMNS,
        "graph_aux_columns": GRAPH_AUX_COLUMNS,
    },
    {
        "experiment_name": "RemoveAmount",
        "source_columns": SOURCE_SCORE_COLUMNS,
        "context_columns": TEMPORAL_CONTEXT_COLUMNS,
        "tabular_aux_columns": TABULAR_AUX_COLUMNS,
        "graph_aux_columns": GRAPH_AUX_COLUMNS,
    },
    {
        "experiment_name": "NoContextAttention",
        "source_columns": SOURCE_SCORE_COLUMNS,
        "context_columns": [],
        "tabular_aux_columns": TABULAR_AUX_COLUMNS,
        "graph_aux_columns": GRAPH_AUX_COLUMNS,
    },
]

if (
    ATTENTION_MODEL_PATH.exists()
    and ATTENTION_SCHEMA_PATH.exists()
    and ATTENTION_ABLATION_PATH.exists()
    and ATTENTION_PREDICTIONS_PATH.exists()
    and not REFRESH_RANKER
):
    attention_ranker_summary = load_json(ATTENTION_SUMMARY_PATH)
    attention_ranker_history = pd.read_csv(ATTENTION_HISTORY_PATH)
    attention_ranker_predictions = pd.read_parquet(ATTENTION_PREDICTIONS_PATH)
    attention_ranker_schema = load_json(ATTENTION_SCHEMA_PATH)
    ablation_df = pd.read_csv(ATTENTION_ABLATION_PATH)
else:
    experiment_results = {}
    ablation_rows = []
    experiment_summaries = []

    for spec in experiment_specs:
        print(f"\nRunning attention experiment: {spec['experiment_name']}")
        result = fit_attention_experiment(
            experiment_name=spec["experiment_name"],
            meta_train_df=meta_train_df,
            rank_valid_df=rank_valid_df,
            rank_test_df=rank_test_df,
            source_score_cols=spec["source_columns"],
            context_cols=spec["context_columns"],
            tabular_aux_cols=spec["tabular_aux_columns"],
            graph_aux_cols=spec["graph_aux_columns"],
        )
        experiment_results[spec["experiment_name"]] = result
        experiment_summaries.append(result["summary"])
        history_slug = slugify(spec["experiment_name"])
        result["history_df"].to_csv(PHASE3_ROOT / f"attention_history_{history_slug}.csv", index=False)
        ablation_rows.append(
            {
                "Configuration": spec["experiment_name"],
                "ValidAUC-PR": result["summary"]["valid_auc_pr"],
                "ValidAUC-ROC": result["summary"]["valid_auc_roc"],
                "AUC-PR": result["summary"]["test_auc_pr"],
                "AUC-ROC": result["summary"]["test_auc_roc"],
                "Precision@100": result["summary"]["test_precision_at_100"],
                "Recall@100": result["summary"]["test_recall_at_100"],
                "NDCG@100": result["summary"]["test_ndcg_at_100"],
                "BestEpoch": result["summary"]["best_epoch"],
            }
        )

    best_experiment_name = max(
        experiment_results,
        key=lambda name: experiment_results[name]["summary"]["valid_auc_pr"],
    )
    print(f"\nSelected exported attention model: {best_experiment_name}")
    main_result = experiment_results[best_experiment_name]

    rank_valid_df["attention_ranker_score"] = main_result["valid_scores"]
    rank_test_df["attention_ranker_score"] = main_result["test_scores"]
    for idx, col in enumerate(SOURCE_SCORE_COLUMNS):
        rank_valid_df[f"{col}_attention"] = main_result["valid_attention"][:, idx]
        rank_test_df[f"{col}_attention"] = main_result["test_attention"][:, idx]

    attention_ranker_predictions = pd.concat([rank_valid_df, rank_test_df], axis=0, ignore_index=True)
    attention_ranker_predictions.to_parquet(ATTENTION_PREDICTIONS_PATH, index=False)

    attention_ranker_history = main_result["history_df"]
    attention_ranker_summary = main_result["summary"]
    attention_ranker_summary["selected_experiment_name"] = best_experiment_name
    attention_ranker_schema = {
        "source_columns": attention_ranker_summary["source_columns"],
        "context_columns": attention_ranker_summary["context_columns"],
        "tabular_aux_columns": attention_ranker_summary["tabular_aux_columns"],
        "graph_aux_columns": attention_ranker_summary["graph_aux_columns"],
        "feature_means": attention_ranker_summary["feature_means"],
        "feature_stds": attention_ranker_summary["feature_stds"],
        "hidden_dim": ATTN_HIDDEN_DIM,
        "dropout": ATTN_DROPOUT,
        "use_graph_branch": attention_ranker_summary["use_graph_branch"],
    }

    attention_ranker_history.to_csv(ATTENTION_HISTORY_PATH, index=False)
    save_json(attention_ranker_summary, ATTENTION_SUMMARY_PATH)
    save_json(attention_ranker_schema, ATTENTION_SCHEMA_PATH)
    save_json(experiment_summaries, ATTENTION_EXPERIMENT_SUMMARIES_PATH)
    torch.save(main_result["model"].state_dict(), ATTENTION_MODEL_PATH)

    ablation_df = pd.DataFrame(ablation_rows).sort_values("ValidAUC-PR", ascending=False).reset_index(drop=True)
    ablation_df.to_csv(ATTENTION_ABLATION_PATH, index=False)

if "attention_ranker_score" not in rank_valid_df.columns:
    rank_valid_df = attention_ranker_predictions[attention_ranker_predictions["split"] == "valid"].copy().reset_index(drop=True)
    rank_test_df = attention_ranker_predictions[attention_ranker_predictions["split"] == "test"].copy().reset_index(drop=True)

attention_ranker_model = rebuild_attention_model(attention_ranker_schema)
attention_ranker_model.load_state_dict(torch.load(ATTENTION_MODEL_PATH, map_location=DEVICE))
attention_ranker_model.eval()

ranking_predictions_df = pd.concat([rank_valid_df, rank_test_df], axis=0, ignore_index=True)
ranking_predictions_df.to_parquet(PHASE3_ROOT / "ranking_predictions.parquet", index=False)

summary_preview = {
    key: value
    for key, value in attention_ranker_summary.items()
    if key not in {"feature_means", "feature_stds"}
}
print(json.dumps(summary_preview, indent=2))
ablation_df


Running attention experiment: FullModel
AttentionRanker | epoch 01 | train_loss=1.0965 | valid_auc_pr=0.4780
AttentionRanker | epoch 02 | train_loss=1.0715 | valid_auc_pr=0.5142
AttentionRanker | epoch 03 | train_loss=1.0471 | valid_auc_pr=0.5370
AttentionRanker | epoch 04 | train_loss=1.0234 | valid_auc_pr=0.5542
AttentionRanker | epoch 05 | train_loss=1.0015 | valid_auc_pr=0.5659
AttentionRanker | epoch 06 | train_loss=0.9800 | valid_auc_pr=0.5737
AttentionRanker | epoch 07 | train_loss=0.9602 | valid_auc_pr=0.5788
AttentionRanker | epoch 08 | train_loss=0.9407 | valid_auc_pr=0.5822
AttentionRanker | epoch 09 | train_loss=0.9222 | valid_auc_pr=0.5845
AttentionRanker | epoch 10 | train_loss=0.9047 | valid_auc_pr=0.5863
AttentionRanker | epoch 11 | train_loss=0.8888 | valid_auc_pr=0.5881
AttentionRanker | epoch 12 | train_loss=0.8713 | valid_auc_pr=0.5902
AttentionRanker | epoch 13 | train_loss=0.8550 | valid_auc_pr=0.5921
AttentionRanker | epoch 14 | train_loss=0.8391 | valid_auc_pr=

,Configuration,ValidAUC-PR,ValidAUC-ROC,AUC-PR,AUC-ROC,Precision@100,Recall@100,NDCG@100,BestEpoch
0,NoContextAttention,0.642443,0.923533,0.578107,0.900708,0.95,0.030814,0.917299,50
1,RemoveNeighborhood,0.620592,0.851650,0.559607,0.819058,0.96,0.031139,0.920901,17
2,RemoveTemporal,0.618368,0.892983,0.525237,0.865115,0.52,0.016867,0.474667,15
3,FullModel,0.616843,0.896843,0.540778,0.871499,0.65,0.021083,0.621992,50
4,RemoveAmount,0.603061,0.895101,0.535315,0.873964,0.71,0.023030,0.647094,10


## Cell 8: Save Raw Ranking Evaluation Outputs

In [11]:
y_test = rank_test_df["y_true"].astype(int).to_numpy()
risk_methods = RANKING_METHODS_BASE + [("AttentionRanker", "attention_ranker_score")]

ranking_efficiency_rows = []
gain_plot_frames = []
for model_name, score_col in risk_methods:
    y_score = rank_test_df[score_col].to_numpy()
    ranking_efficiency_rows.append(
        {
            "Model": model_name,
            "ScoreColumn": score_col,
            "Precision@100": precision_at_k(y_test, y_score, 100),
            "Recall@100": recall_at_k(y_test, y_score, 100),
            "NDCG@100": ndcg_at_k(y_test, y_score, 100),
            "AUC-PR": float(average_precision_score(y_test, y_score)),
            "AUC-ROC": float(roc_auc_score(y_test, y_score)),
        }
    )
    curve_df = cumulative_gain_curve(y_test, y_score)
    curve_df["Model"] = model_name
    curve_df["ScoreColumn"] = score_col
    gain_plot_frames.append(curve_df)

ranking_efficiency_df = pd.DataFrame(ranking_efficiency_rows).sort_values("NDCG@100", ascending=False)
gain_curve_df = pd.concat(gain_plot_frames, axis=0, ignore_index=True)

ranking_efficiency_df.to_csv(PHASE3_ROOT / "ranking_efficiency_raw.csv", index=False)
gain_curve_df.to_csv(PHASE3_ROOT / "ranking_gain_curve_raw.csv", index=False)

attention_heatmap_df = (
    rank_test_df.sort_values("attention_ranker_score", ascending=False)
    .head(ATTENTION_HEATMAP_CASES)
    [[ID_COL, "attention_ranker_score"] + [f"{col}_attention" for col in SOURCE_SCORE_COLUMNS]]
    .copy()
)
attention_heatmap_df.to_csv(PHASE3_ROOT / "attention_heatmap_cases_raw.csv", index=False)

## Cell 9: Save Raw Case-Level Explanations

In [12]:
top_alerts_df = (
    rank_test_df.sort_values("attention_ranker_score", ascending=False)
    .head(TOP_ALERT_CASES)
    .copy()
    .reset_index(drop=True)
)
top_alert_ids = top_alerts_df[ID_COL].astype(int).tolist()
top_alert_feature_df = top_alerts_df[[ID_COL, "attention_ranker_score", "y_true"]].merge(
    test_feature_lookup,
    on=ID_COL,
    how="left",
)

case_lookup = {
    int(row[ID_COL]): {
        TIME_COL: int(row[TIME_COL]),
        "y_true": int(row["y_true"]),
        "attention_ranker_score": float(row["attention_ranker_score"]),
    }
    for _, row in top_alerts_df.iterrows()
}

if RUN_SHAP:
    lgbm_case_feature_df, top_alert_shap_values = top_lgbm_features_for_cases(
        case_feature_df=top_alert_feature_df,
        case_ids=top_alert_ids,
        case_lookup=case_lookup,
        feature_schema=feature_schema,
        lgbm_model=lgbm_model,
    )
else:
    lgbm_case_feature_df = pd.DataFrame()
    top_alert_shap_values = np.empty((0, 0))

if RUN_GRAPH_CASE_EXPLANATIONS:
    graph_case_evidence_df = explain_top_cases_with_graph_context(
        case_df=top_alerts_df[[ID_COL, TIME_COL, "attention_ranker_score", "y_true"]],
        graph_lookup_df=graph_lookup_df,
        edge_df=full_edge_df,
        entity_df=full_entity_df,
    )
else:
    graph_case_evidence_df = pd.DataFrame()

case_summary_df = build_case_summary_table(
    top_alerts_df,
    lgbm_case_feature_df,
    graph_case_evidence_df,
    SOURCE_SCORE_COLUMNS,
    SOURCE_DISPLAY_NAMES,
)

top_alerts_df.to_csv(PHASE3_ROOT / "top_alert_cases_raw.csv", index=False)
lgbm_case_feature_df.to_csv(PHASE3_ROOT / "lgbm_case_feature_explanations_raw.csv", index=False)
graph_case_evidence_df.to_csv(PHASE3_ROOT / "graph_case_evidence_raw.csv", index=False)
case_summary_df.to_csv(PHASE3_ROOT / "case_summary_raw.csv", index=False)

reference_case = choose_reference_case(top_alerts_df)
reference_txn_id = int(reference_case[ID_COL])
reference_feature_rows = (
    lgbm_case_feature_df[lgbm_case_feature_df[ID_COL] == reference_txn_id]
    .sort_values("FeatureRank")
    .copy()
)
reference_graph_rows = (
    graph_case_evidence_df[graph_case_evidence_df[ID_COL] == reference_txn_id]
    .sort_values("EdgeWeight", ascending=False)
    .copy()
)
reference_attention_df = pd.DataFrame(
    {
        "Source": SOURCE_DISPLAY_NAMES,
        "AttentionWeight": [float(reference_case[f"{col}_attention"]) for col in SOURCE_SCORE_COLUMNS],
        "TransactionID": reference_txn_id,
        "TrueLabel": int(reference_case["y_true"]),
    }
)
reference_feature_rows.to_csv(PHASE3_ROOT / "reference_case_lgbm_features_raw.csv", index=False)
reference_graph_rows.to_csv(PHASE3_ROOT / "reference_case_graph_evidence_raw.csv", index=False)
reference_attention_df.to_csv(PHASE3_ROOT / "reference_case_attention_raw.csv", index=False)

## Cell 10: Save Raw Explanation Fidelity And Robustness Outputs

In [13]:
explanation_sample_df = (
    rank_test_df.sort_values("attention_ranker_score", ascending=False)
    .head(EXPLANATION_SAMPLE_SIZE)
    .copy()
    .reset_index(drop=True)
)
explanation_feature_df = explanation_sample_df[[ID_COL]].merge(test_feature_lookup, on=ID_COL, how="left")
explanation_feature_df = explanation_feature_df[feature_schema["feature_columns"]].copy()

if RUN_SHAP and RUN_EXPLANATION_EVALUATION:
    shap_explainer = shap.TreeExplainer(lgbm_model.booster_)
    explanation_shap_values = shap_explainer.shap_values(explanation_feature_df)
    if isinstance(explanation_shap_values, list):
        explanation_shap_values = explanation_shap_values[-1]
else:
    explanation_shap_values = np.empty((0, 0))

feature_means = attention_ranker_schema["feature_means"]
feature_stds = attention_ranker_schema["feature_stds"]
attention_eval_df = explanation_sample_df.copy()
standardize_cols = []
for col_group in [
    attention_ranker_schema["context_columns"],
    attention_ranker_schema["tabular_aux_columns"],
    attention_ranker_schema["graph_aux_columns"],
]:
    for col in col_group:
        if col not in standardize_cols:
            standardize_cols.append(col)

if standardize_cols:
    attention_eval_df = apply_standardization(
        attention_eval_df,
        standardize_cols,
        feature_means,
        feature_stds,
    )

attention_source_array, attention_context_array, attention_tabular_aux_array, attention_graph_aux_array = build_attention_arrays(
    attention_eval_df,
    attention_ranker_schema["source_columns"],
    attention_ranker_schema["context_columns"],
    attention_ranker_schema["tabular_aux_columns"],
    attention_ranker_schema["graph_aux_columns"],
)
attention_scores, attention_weights = run_attention_forward(
    attention_ranker_model,
    attention_source_array,
    attention_context_array,
    attention_tabular_aux_array,
    attention_graph_aux_array,
)

if RUN_EXPLANATION_EVALUATION:
    medians = test_df[feature_schema["feature_columns"]].median(numeric_only=True)
    medians = medians.reindex(feature_schema["feature_columns"]).fillna(0.0)

    lgbm_mask_curve_df, lgbm_base_scores = compute_lgbm_mask_curve(
        feature_df=explanation_feature_df,
        shap_values=explanation_shap_values,
        medians=medians,
        feature_schema=feature_schema,
        lgbm_model=lgbm_model,
        max_k=4,
    )
    attention_mask_curve_df, attention_base_scores = compute_attention_mask_curve(
        model=attention_ranker_model,
        source_array=attention_source_array,
        context_array=attention_context_array,
        tabular_aux_array=attention_tabular_aux_array,
        graph_aux_array=attention_graph_aux_array,
        attention_weights=attention_weights,
        max_k=4,
    )

    lgbm_top_retention = lgbm_mask_curve_df[
        (lgbm_mask_curve_df["Method"] == "LightGBM")
        & (lgbm_mask_curve_df["MaskingStrategy"] == "Top")
        & (lgbm_mask_curve_df["ComponentsMasked"] == 1)
    ]["ScoreRetention"].iloc[0]
    lgbm_random_retention = lgbm_mask_curve_df[
        (lgbm_mask_curve_df["Method"] == "LightGBM")
        & (lgbm_mask_curve_df["MaskingStrategy"] == "Random")
        & (lgbm_mask_curve_df["ComponentsMasked"] == 1)
    ]["ScoreRetention"].iloc[0]
    attention_top_retention = attention_mask_curve_df[
        (attention_mask_curve_df["Method"] == "AttentionRanker")
        & (attention_mask_curve_df["MaskingStrategy"] == "Top")
        & (attention_mask_curve_df["ComponentsMasked"] == 1)
    ]["ScoreRetention"].iloc[0]
    attention_random_retention = attention_mask_curve_df[
        (attention_mask_curve_df["Method"] == "AttentionRanker")
        & (attention_mask_curve_df["MaskingStrategy"] == "Random")
        & (attention_mask_curve_df["ComponentsMasked"] == 1)
    ]["ScoreRetention"].iloc[0]

    fidelity_df = pd.DataFrame(
        [
            {
                "Method": "LightGBM",
                "MeanTopMaskedScoreDrop": float(np.mean(lgbm_base_scores) * (1 - lgbm_top_retention)),
                "MeanRandomMaskedScoreDrop": float(np.mean(lgbm_base_scores) * (1 - lgbm_random_retention)),
            },
            {
                "Method": "AttentionRanker",
                "MeanTopMaskedScoreDrop": float(np.mean(attention_base_scores) * (1 - attention_top_retention)),
                "MeanRandomMaskedScoreDrop": float(np.mean(attention_base_scores) * (1 - attention_random_retention)),
            },
        ]
    )
    fidelity_df["TopVsRandomDropRatio"] = (
        fidelity_df["MeanTopMaskedScoreDrop"] / fidelity_df["MeanRandomMaskedScoreDrop"].replace(0, np.nan)
    )
else:
    fidelity_df = pd.DataFrame()
    lgbm_mask_curve_df = pd.DataFrame()
    attention_mask_curve_df = pd.DataFrame()

perturbation_curve_df = pd.concat([lgbm_mask_curve_df, attention_mask_curve_df], axis=0, ignore_index=True)
fidelity_df.to_csv(PHASE3_ROOT / "explanation_fidelity_raw.csv", index=False)
perturbation_curve_df.to_csv(PHASE3_ROOT / "perturbation_curves_raw.csv", index=False)

attention_size_support = compute_attention_support_size(attention_weights, threshold=0.80)
lightgbm_size_rows = []
if RUN_SHAP and RUN_EXPLANATION_EVALUATION:
    abs_shap = np.abs(explanation_shap_values)
    for row in abs_shap:
        ordered = np.sort(row)[::-1]
        total = max(float(np.sum(ordered)), 1e-8)
        support_size = int(np.searchsorted(np.cumsum(ordered) / total, 0.80) + 1)
        lightgbm_size_rows.append(support_size)
lightgbm_size_support = np.array(lightgbm_size_rows, dtype=int) if lightgbm_size_rows else np.array([], dtype=int)

support_size_df = pd.DataFrame(
    {
        "SupportSize": np.concatenate([lightgbm_size_support, attention_size_support]),
        "Method": (["LightGBM"] * len(lightgbm_size_support)) + (["AttentionRanker"] * len(attention_size_support)),
    }
)
support_size_df.to_csv(PHASE3_ROOT / "explanation_support_sizes_raw.csv", index=False)

noise_rows = []
for noise_level in NOISE_LEVELS:
    if noise_level == 0.0:
        noise_rows.append(
            {
                "NoiseLevel": float(noise_level),
                "ScoreCorrelation": 1.0,
                "Top1AttentionAgreement": 1.0,
                "MeanAbsScoreShift": 0.0,
            }
        )
        continue

    noisy_source = attention_source_array.copy()
    noisy_context = attention_context_array.copy()
    noisy_tabular_aux = attention_tabular_aux_array.copy()
    noisy_graph_aux = attention_graph_aux_array.copy()

    # Perturb only the raw score channel, then recompute centered scores so the
    # two-channel source representation remains internally consistent.
    noisy_raw_scores = noisy_source[:, :, 0] + np.random.normal(
        0.0,
        noise_level,
        size=noisy_source[:, :, 0].shape,
    ).astype(np.float32)
    noisy_raw_scores = np.clip(noisy_raw_scores, 0.0, 1.0)
    noisy_source[:, :, 0] = noisy_raw_scores
    noisy_source[:, :, 1] = noisy_raw_scores - noisy_raw_scores.mean(axis=1, keepdims=True)

    # Only perturb real feature groups. For experiments like NoContextAttention,
    # the context branch is a placeholder zero column and should not be noised.
    if attention_ranker_schema["context_columns"]:
        noisy_context = noisy_context + np.random.normal(
            0.0,
            noise_level,
            size=noisy_context.shape,
        ).astype(np.float32)
    if attention_ranker_schema["tabular_aux_columns"]:
        noisy_tabular_aux = noisy_tabular_aux + np.random.normal(
            0.0,
            noise_level,
            size=noisy_tabular_aux.shape,
        ).astype(np.float32)
    if attention_ranker_schema["graph_aux_columns"]:
        noisy_graph_aux = noisy_graph_aux + np.random.normal(
            0.0,
            noise_level,
            size=noisy_graph_aux.shape,
        ).astype(np.float32)

    noisy_scores, noisy_attention = run_attention_forward(
        attention_ranker_model,
        noisy_source,
        noisy_context,
        noisy_tabular_aux,
        noisy_graph_aux,
    )
    noise_rows.append(
        {
            "NoiseLevel": float(noise_level),
            "ScoreCorrelation": float(np.corrcoef(attention_scores, noisy_scores)[0, 1]),
            "Top1AttentionAgreement": float(
                np.mean(np.argmax(attention_weights, axis=1) == np.argmax(noisy_attention, axis=1))
            ),
            "MeanAbsScoreShift": float(np.mean(np.abs(attention_scores - noisy_scores))),
        }
    )

noise_robustness_df = pd.DataFrame(noise_rows).sort_values("NoiseLevel")
noise_robustness_df.to_csv(PHASE3_ROOT / "noise_robustness_raw.csv", index=False)

## Cell 11: Save Raw Operational Evaluation Outputs

In [14]:
simulation_rows = []
for model_name, score_col in risk_methods:
    sim_df = simulate_daily_budget(rank_test_df, score_col, DAILY_BUDGETS)
    if sim_df.empty:
        continue
    sim_df["Model"] = model_name
    simulation_rows.append(sim_df)

simulation_daily_df = pd.concat(simulation_rows, axis=0, ignore_index=True)
budget_performance_df = (
    simulation_daily_df.groupby(["Budget", "Model", "ScoreColumn"], as_index=False)
    .agg(
        AvgDailyRecall=("DailyRecall", "mean"),
        AvgDailyPrecision=("DailyPrecision", "mean"),
        AvgFraudCaptured=("FraudCaptured", "mean"),
    )
)

if RUN_ROI_ANALYSIS:
    roi_df = budget_performance_df.copy()
    roi_df["AvgGrossBenefit"] = roi_df["AvgFraudCaptured"] * AVG_FRAUD_VALUE_CAPTURED
    roi_df["AvgReviewCost"] = roi_df["Budget"] * REVIEW_COST_PER_ALERT
    roi_df["AvgNetBenefit"] = roi_df["AvgGrossBenefit"] - roi_df["AvgReviewCost"]
    roi_df["ROI"] = roi_df["AvgNetBenefit"] / roi_df["AvgReviewCost"].replace(0, np.nan)
else:
    roi_df = pd.DataFrame()

simulation_daily_df.to_csv(PHASE3_ROOT / "daily_budget_simulation_raw.csv", index=False)
budget_performance_df.to_csv(PHASE3_ROOT / "budget_performance_raw.csv", index=False)
roi_df.to_csv(PHASE3_ROOT / "roi_analysis_raw.csv", index=False)

## Cell 12: Save Internal Manifest

In [15]:
phase3_manifest = {
    "phase": "phase03_ranking_explainability",
    "title_alignment": "Explainable Fraud Detection and Alert Prioritization Using Tabular and Graph-Informed Models",
    "design": {
        "phase_role": "raw_output_only",
        "frozen_inputs": ["phase01_data_baselines", "phase02_graph_models"],
        "selected_experiment_name": attention_ranker_summary.get("selected_experiment_name", attention_ranker_summary["experiment_name"]),
    },
    "attention_experiments": [spec["experiment_name"] for spec in experiment_specs],
    "files": {
        "phase3_master_frame": str(PHASE3_ROOT / "phase3_master_frame.parquet"),
        "phase3_meta_train_frame": str(PHASE3_ROOT / "phase3_meta_train_frame.parquet"),
        "heuristic_weights": str(PHASE3_ROOT / "heuristic_weights.json"),
        "ranking_predictions": str(PHASE3_ROOT / "ranking_predictions.parquet"),
        "attention_ranker_predictions": str(ATTENTION_PREDICTIONS_PATH),
        "attention_ranker_state": str(ATTENTION_MODEL_PATH),
        "attention_ranker_schema": str(ATTENTION_SCHEMA_PATH),
        "attention_ranker_summary": str(ATTENTION_SUMMARY_PATH),
        "attention_ranker_history": str(ATTENTION_HISTORY_PATH),
        "attention_experiment_summaries": str(ATTENTION_EXPERIMENT_SUMMARIES_PATH),
        "attention_ablation_raw": str(ATTENTION_ABLATION_PATH),
        "ranking_efficiency_raw": str(PHASE3_ROOT / "ranking_efficiency_raw.csv"),
        "ranking_gain_curve_raw": str(PHASE3_ROOT / "ranking_gain_curve_raw.csv"),
        "attention_heatmap_cases_raw": str(PHASE3_ROOT / "attention_heatmap_cases_raw.csv"),
        "top_alert_cases_raw": str(PHASE3_ROOT / "top_alert_cases_raw.csv"),
        "lgbm_case_feature_explanations_raw": str(PHASE3_ROOT / "lgbm_case_feature_explanations_raw.csv"),
        "graph_case_evidence_raw": str(PHASE3_ROOT / "graph_case_evidence_raw.csv"),
        "case_summary_raw": str(PHASE3_ROOT / "case_summary_raw.csv"),
        "reference_case_lgbm_features_raw": str(PHASE3_ROOT / "reference_case_lgbm_features_raw.csv"),
        "reference_case_graph_evidence_raw": str(PHASE3_ROOT / "reference_case_graph_evidence_raw.csv"),
        "reference_case_attention_raw": str(PHASE3_ROOT / "reference_case_attention_raw.csv"),
        "explanation_fidelity_raw": str(PHASE3_ROOT / "explanation_fidelity_raw.csv"),
        "perturbation_curves_raw": str(PHASE3_ROOT / "perturbation_curves_raw.csv"),
        "explanation_support_sizes_raw": str(PHASE3_ROOT / "explanation_support_sizes_raw.csv"),
        "noise_robustness_raw": str(PHASE3_ROOT / "noise_robustness_raw.csv"),
        "daily_budget_simulation_raw": str(PHASE3_ROOT / "daily_budget_simulation_raw.csv"),
        "budget_performance_raw": str(PHASE3_ROOT / "budget_performance_raw.csv"),
        "roi_analysis_raw": str(PHASE3_ROOT / "roi_analysis_raw.csv"),
    },
    "proposal_artifacts_to_generate_later": {
        "rq2": [
            "table_2_1_ranking_efficiency.csv",
            "table_2_2_ranking_component_ablation.csv",
            "figure_2_1_ranking_performance_curve",
            "figure_2_2_learned_attention_weights",
        ],
        "rq3": [
            "table_3_1_human_evaluation.csv",
            "table_3_2_case_based_analysis.csv",
            "figure_3_1_explanation_comparison",
            "figure_3_2_analyst_confidence_distribution",
        ],
        "rq4": [
            "table_4_1_explanation_fidelity.csv",
            "table_4_2_robustness_across_noise_levels.csv",
            "figure_4_1_perturbation_analysis",
            "figure_4_2_explanation_size_distribution",
        ],
        "rq6": [
            "table_6_1_budget_constrained_performance.csv",
            "table_6_2_roi_analysis.csv",
            "figure_6_1_effort_vs_recall_frontier",
            "figure_6_2_daily_simulation",
        ],
    },
}
save_json(phase3_manifest, OUTPUT_ROOT / "manifests" / "phase03_artifact_manifest.json")
save_json(phase3_manifest, PHASE3_ROOT / "phase03_artifact_manifest.json")
phase3_manifest

{'phase': 'phase03_ranking_explainability',
 'title_alignment': 'Explainable Fraud Detection and Alert Prioritization Using Tabular and Graph-Informed Models',
 'design': {'phase_role': 'raw_output_only',
  'frozen_inputs': ['phase01_data_baselines', 'phase02_graph_models'],
  'selected_experiment_name': 'NoContextAttention'},
 'attention_experiments': ['FullModel',
  'RemoveNeighborhood',
  'RemoveTemporal',
  'RemoveAmount',
  'NoContextAttention'],
 'files': {'phase3_master_frame': '/kaggle/working/thesis_outputs/artifacts/phase03_ranking_explainability/phase3_master_frame.parquet',
  'phase3_meta_train_frame': '/kaggle/working/thesis_outputs/artifacts/phase03_ranking_explainability/phase3_meta_train_frame.parquet',
  'heuristic_weights': '/kaggle/working/thesis_outputs/artifacts/phase03_ranking_explainability/heuristic_weights.json',
  'ranking_predictions': '/kaggle/working/thesis_outputs/artifacts/phase03_ranking_explainability/ranking_predictions.parquet',
  'attention_ranker_pr

## Cell 13: Hand-Off To `rq*_artifacts.py`
#
Later reporting scripts should read:
- `ranking_predictions.parquet`
- `attention_ablation_raw.csv`
- `attention_heatmap_cases_raw.csv`
- `case_summary_raw.csv`
- `reference_case_*_raw.csv`
- `explanation_fidelity_raw.csv`
- `perturbation_curves_raw.csv`
- `explanation_support_sizes_raw.csv`
- `noise_robustness_raw.csv`
- `budget_performance_raw.csv`
- `roi_analysis_raw.csv`

In [16]:
print("Phase 03 finished.")
print("Phase 03 now saves raw ranking, explainability, robustness, and operational outputs only.")
print("Generate proposal-facing RQ2, RQ3, RQ4, and RQ6 artifacts separately after reviewing these frozen outputs.")

Phase 03 finished.
Phase 03 now saves raw ranking, explainability, robustness, and operational outputs only.
Generate proposal-facing RQ2, RQ3, RQ4, and RQ6 artifacts separately after reviewing these frozen outputs.
